# LIBRARIES

In [ ]:
import math
import numpy as np
import pandas as pd
import json

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, train_test_split

# FUNCTIONS

In [ ]:
# Timestamp Transformation:
def create_sequences(data, target_col, sequence_length):
    sequences = []
    targets = []

    target = data[target_col]
    data = data.drop(target_col, axis=1)

    for i in range(len(data) - sequence_length + 1):
        sequences.append(data[i:i+sequence_length])
        targets.append(target[i:i+sequence_length])
    return np.array(sequences), np.array(targets)

In [ ]:
# Reverse Min Max Scaler;
def invert_min_max(X_scaled, X_max, X_min, range_max=1, range_min=0):
    return (X_scaled-range_min)/(range_max-range_min)*(X_max-X_min) + X_min

In [ ]:
#Nature encoder;
def nature_encode(df: pd.DataFrame, col: str, div_period: int):
    """
    Applies a Nature Cyclical Transformation, where each period
    is a combination of sin and cos.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame on which the function will be applied.
    col : str
        Period column on which the function will be applied.
    div_period : int
        Amount of periods until the cycle restarts (e.g. month=12, week=7, etc).
    """

    df[col + "_sin"] = np.sin(2 * np.pi * df[col] / div_period)
    df[col + "_cos"] = np.cos(2 * np.pi * df[col] / div_period)
    return None

In [ ]:
# Custom RMSE metrics
def rmse(y_true, y_pred):
  import tensorflow as tf
  from tensorflow.keras import backend as K
  return K.sqrt(K.mean(K.square(y_pred - y_true)))

In [ ]:
# Bidirectional LSTM:
def bidirectional_lstm_model(X_train, y_train, X_val, y_val, batch_size, epochs):
  from tensorflow.keras.models import Sequential
  from tensorflow.keras.layers import Bidirectional, LSTM, Dense, Dropout
  from tensorflow.keras.optimizers import Adam

  learning_rate=0.001
  steps_per_epoch = math.ceil(len(X_train) / batch_size)
  validation_steps = math.ceil(len(X_val) / batch_size)

  #Adicionar Dropout
  model = Sequential()
  model.add(Bidirectional(LSTM(128, activation='relu', return_sequences=False), input_shape=(X_train.shape[1], X_train.shape[2])))
  model.add(Dropout(0.1)),
  model.add(Dense(1))

  print(f'Steps per epoch: {steps_per_epoch}')
  print(f'Validation steps: {validation_steps}')

  optimizer = Adam(learning_rate=learning_rate)
  model.compile(loss=rmse, optimizer=optimizer)

  history = model.fit(X_train, y_train,
                      epochs=epochs,
                      #steps_per_epoch=steps_per_epoch,
                      #validation_steps=validation_steps,
                      batch_size=batch_size,
                      validation_data=(X_val, y_val),
                      verbose=1)

  return model, history

# DATA SELECTION

# DATA PROCESSING

In [ ]:
# Defining constants:
num_timesteps = 7
target_col = 'target'

In [ ]:
nature_encode(df, 'weekday', 7)
nature_encode(df, 'month', 12)
nature_encode(df, 'day', 31)
df = df.drop(['weekday', 'day', 'month', 'year'], axis=1)

The Max and Min values of the target will have to be reserved for the Inverse Min Max Scaler that will be applied in the prediction results.

In [ ]:
# Min Max variables
target_max = df[target_col].max();
target_min = df[target_col].min();

In [ ]:
# Min Max Scaler:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df[['target', 'buy_dollar', 'ipca', 'selic']] = scaler.fit_transform(df[['target', 'buy_dollar', 'ipca', 'selic']]) #Aplied on all columns;
df

### Train and Test Division:
The Test data will be reserved to create predictions after the model is trained and validated. These predictions will occur in diffente time ranges, defined by the *thresholds* array. For the data to be trully separeted from the training and validation data, the maximum range of prediction will be cutted from the entire dataset.


The maximum reserved values for the prediction must be equal to the biggest threshold and need to consider the effects of the *create_sequences()* function. This function will by nature decrease the size of any dataset by the number of timesteps (num_timesteps) minus one. Therefore, to keep the prediction thresholds intact, these values need to be added to the Train and test Division.

In [ ]:
# Train and Test Maximum Division;
df_test = df.sort_index().iloc[-(max_threshold + num_timesteps - 1):] #Maximum reserved values for prediction;
df = df.sort_index().iloc[:-(max_threshold + num_timesteps - 1)] #Excluding the test values from the rest of the dataframe;
len(df_test)

### Train and Validation Split


In [ ]:
# Train and Validation Split;
train_data, val_data= np.split(df, [int(.70 *len(df))])

print("Original data size:", len(df))
print("Train data size:", len(train_data))
print("Validation data size", len(val_data))

In [ ]:
# Feature and Target division + Timestep added;
X_train, y_train = create_sequences(train_data, target_col, num_timesteps)
X_val, y_val = create_sequences(val_data, target_col, num_timesteps)
X_test, y_test = create_sequences(df_test, target_col, num_timesteps)

In [ ]:
# Defining constants:
num_features = X_train.shape[2]
num_samples = X_train.shape[0]

In [ ]:
# Check data division;
print("Shape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_val:", X_val.shape)
print("Shape of y_val:", y_val.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)

In [ ]:
# Train model;
model, history = bidirectional_lstm_model(X_train, y_train, X_val, y_val, batch_size=32, epochs=50)

print("=============================")
model.summary()
print("=============================")

In [ ]:
# Limit test dataset based on the size of the timesteps;
df_test = df_test.iloc[:-(num_timesteps-1)]

In [ ]:
prediction = model.predict(X_test)
prediction

In [ ]:
prediction = pd.DataFrame(prediction)
prediction.columns = [target_col]
prediction.index = df_test.index

In [ ]:
# Invert the scaling for target_col;
prediction[target_col] = prediction[target_col].apply(lambda x: invert_min_max(x, target_max, target_min))
df_test[target_col] = df_test[target_col].apply(lambda x: invert_min_max(x, target_max, target_min))
prediction, df_test

In [ ]:
from sklearn.metrics import mean_squared_error
rmse = mean_squared_error(df_test[target_col], prediction[target_col], squared=False)
print("RMSE:", rmse)

In [ ]:
df_test[target_col].plot(figsize=(10,5), legend=True)
prediction[target_col].plot(figsize=(10,5), legend=True)

In [ ]:
# #Determining best batch size;

# from math import ceil
# import numpy as np
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Bidirectional, LSTM, Dense, Dropout

# batch_sizes = [2, 4, 8, 16, 32, 64, 128]

# for batch_size in batch_sizes:
#     steps_per_epoch = ceil(len(X_train) / batch_size)
#     validation_steps = ceil(len(X_val) / batch_size)

#     print(f'Testing batch_size: {batch_size}')

#     # Model definition (simplified for testing)
#     model = Sequential([
#       Bidirectional(LSTM(64, activation='relu', return_sequences=True), input_shape=(num_timesteps, num_features)),
#       Dropout(0.1),
#       Bidirectional(LSTM(32)),
#       Dropout(0.1),
#       Dense(1)
#     ])

#     model.compile(optimizer='adam', loss='mse', metrics=['mse'])

#     # Train the model
#     history = model.fit(X_train, y_train,
#                         epochs=3,  # Use fewer epochs for quick testing
#                         batch_size=batch_size,
#                         steps_per_epoch=steps_per_epoch,
#                         validation_steps=validation_steps,
#                         validation_data=(X_val, y_val))

#     # Print the results
#     print(f'Batch size: {batch_size}')
#     print(f'Training loss: {history.history["loss"][-1]}, Validation loss: {history.history["val_loss"][-1]}')
# #     print('='*50)